In [2]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad

In [68]:
DATA_DIR = "/home/kchen/microbiome/gut_microbiome_GPT/datasets/hmc_final/"
pretrain = ad.read_h5ad(DATA_DIR + "pretrain.h5ad")
downstream_train = ad.read_h5ad(DATA_DIR + "downstream_train.h5ad")
downstream_test = ad.read_h5ad(DATA_DIR + "downstream_test.h5ad")

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [69]:
temp_taxa = ad.read_h5ad(DATA_DIR + "temp_taxa.h5ad")

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [70]:
# Split the taxa names by "." and count the maximum number of levels

n = 0
n2 = 2
level_distributions = {}
taxa_split = pretrain.var['taxa'].str.split('.')
for i in range(0, 8):
    level_distributions[i] = taxa_split.str[i].value_counts()
    count_more_than_n = (level_distributions[i] > n).sum()
    count_more_than_n2 = (level_distributions[i] > n2).sum()
    # print(f"Distribution of {i} level taxa:")
    # print(level_distribution)
    print(f"Number of taxa with more than {n} occurrences at level {i}: {count_more_than_n}")
    print(f"Number of taxa with more than {n2} occurrences at level {i}: {count_more_than_n2}")
    print(f"difference: {count_more_than_n - count_more_than_n2}")
    

print(taxa_split.apply(len).value_counts().sort_index())

Number of taxa with more than 0 occurrences at level 0: 4
Number of taxa with more than 2 occurrences at level 0: 2
difference: 2
Number of taxa with more than 0 occurrences at level 1: 53
Number of taxa with more than 2 occurrences at level 1: 28
difference: 25
Number of taxa with more than 0 occurrences at level 2: 111
Number of taxa with more than 2 occurrences at level 2: 46
difference: 65
Number of taxa with more than 0 occurrences at level 3: 217
Number of taxa with more than 2 occurrences at level 3: 80
difference: 137
Number of taxa with more than 0 occurrences at level 4: 353
Number of taxa with more than 2 occurrences at level 4: 135
difference: 218
Number of taxa with more than 0 occurrences at level 5: 1066
Number of taxa with more than 2 occurrences at level 5: 2
difference: 1064
Number of taxa with more than 0 occurrences at level 6: 2
Number of taxa with more than 2 occurrences at level 6: 0
difference: 2
Number of taxa with more than 0 occurrences at level 7: 0
Number o

In [71]:
# Define a function to assign categories based on the period_taxa DataFrame
def assign_categories(taxa_name, period_taxa=pd.Series([], dtype=object)):
    split_taxa = taxa_name.split(".")
    if len(split_taxa) > 6:
        # Use the period_taxa Series to assign categories properly
        if taxa_name not in period_taxa.index:
            raise ValueError(f"Taxa '{taxa_name}' not found in period_taxa.")
        return period_taxa.loc[taxa_name]
    else:
        # Assign generic categories for taxa with 6 or fewer levels
        return split_taxa

# # Apply the function to the taxa names and create a new varm
# taxon_lists = data.var["taxa"].apply(assign_categories, period_taxa=period_taxa)
# categories = ["Domain", "Phylum", "Class", "Order", "Family", "Genus"]

# taxon_df = pd.DataFrame(taxon_lists.tolist(), index=data.var_names, columns=categories)
# data.varm['taxonomy'] = taxon_df

In [ ]:
def copy_taxonomy(data, temp_taxa):
    diff = data.var_names.difference(temp_taxa.var_names)
    shared = data.var_names.intersection(temp_taxa.var_names)

    tax_temp = np.asarray(temp_taxa.varm["taxonomy"], dtype=object)

    # output DF indexed by data taxa
    # temp taxonomy as DataFrame, indexed by taxa
    tax_temp = pd.DataFrame(
        temp_taxa.varm["taxonomy"],
        index=temp_taxa.var_names
    )
    tax_out = pd.DataFrame(
        index=data.var_names,
        columns=tax_temp.columns,
        dtype=object
    )
    # fill shared directly
    tax_out.loc[shared] = tax_temp.loc[shared]

    # fill missing via assign_categories
    for t in diff:
        tax_out.loc[t] = assign_categories(t)
        
    # store back into AnnData (varm expects array-like)
    data.varm["taxonomy"] = tax_out

In [84]:
copy_taxonomy(pretrain, temp_taxa)
copy_taxonomy(downstream_train, temp_taxa)
copy_taxonomy(downstream_test, temp_taxa)

Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia: .
Mismatch for Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella: .
Mismatch for Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.NA: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia: .
Mismatch for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegali

In [101]:
downstream_test

AnnData object with n_obs × n_vars = 6090 × 1514
    obs: 'drr', 'study_id', 'region', 'total_bases', 'instrument', 'downstream_task', 'categorical_label', 'continuous_label'
    varm: 'taxonomy'

In [54]:
downstream_test.varm["taxonomy"]

,Domain,Phylum,Class,Order,Family,Genus
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Senegalimassilia
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria,Bacillota,Bacilli,Erysipelotrichales,Erysipelotrichaceae,Holdemanella
...,...,...,...,...,...,...
Bacteria.Marinimicrobia (SAR406 clade).Incertae Sedis.Incertae Sedis.Incertae Sedis.Incertae Sedis,Bacteria,Marinimicrobia (SAR406 clade),Incertae Sedis,Incertae Sedis,Incertae Sedis,Incertae Sedis
Bacteria.Balneolota.Balneolia.Balneolales.Balneolaceae.Balneola,Bacteria,Balneolota,Balneolia,Balneolales,Balneolaceae,Balneola
Bacteria.Pseudomonadota.Gammaproteobacteria.Thiomicrospirales.Thiomicrospiraceae.Thiomicrospira,Bacteria,Pseudomonadota,Gammaproteobacteria,Thiomicrospirales,Thiomicrospiraceae,Thiomicrospira
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Crocinitomicaceae.Brumimicrobium,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Crocinitomicaceae,Brumimicrobium


In [103]:
for name, row in zip(downstream_test.var_names, downstream_test.varm["taxonomy"].iterrows()):
    if ".".join(row[1]) != name:
        print(f"Mismatch for {name}: {row[1]}")
    else:
        print(f"Match for {name}")
    # print("." . join(row[1]), name)

Match for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter
Match for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella
Match for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia
Match for Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia
Match for Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella
Match for Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.NA
Match for Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Lactiplantibacillus
Match for Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Limosilactobacillus
Match for Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Pediococcus
Match for Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Weissella
Match for Bacteria.Bacillota.Bacilli.Lactobacillales.Streptococcaceae.Streptococcus
Match for Bact

In [89]:
downstream_test.varm["taxonomy"]

,Domain,Phylum,Class,Order,Family,Genus
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Senegalimassilia
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria,Bacillota,Bacilli,Erysipelotrichales,Erysipelotrichaceae,Holdemanella
...,...,...,...,...,...,...
Bacteria.Marinimicrobia (SAR406 clade).Incertae Sedis.Incertae Sedis.Incertae Sedis.Incertae Sedis,Bacteria,Marinimicrobia (SAR406 clade),Incertae Sedis,Incertae Sedis,Incertae Sedis,Incertae Sedis
Bacteria.Balneolota.Balneolia.Balneolales.Balneolaceae.Balneola,Bacteria,Balneolota,Balneolia,Balneolales,Balneolaceae,Balneola
Bacteria.Pseudomonadota.Gammaproteobacteria.Thiomicrospirales.Thiomicrospiraceae.Thiomicrospira,Bacteria,Pseudomonadota,Gammaproteobacteria,Thiomicrospirales,Thiomicrospiraceae,Thiomicrospira
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Crocinitomicaceae.Brumimicrobium,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Crocinitomicaceae,Brumimicrobium


In [104]:
pretrain.write_h5ad(DATA_DIR + "pretrain.h5ad")
downstream_train.write_h5ad(DATA_DIR + "downstream_train.h5ad")
downstream_test.write_h5ad(DATA_DIR + "downstream_test.h5ad")

In [ ]:
# # Read the period_taxa.csv file
# period_taxa = pd.read_csv(DATA_DIR + "period_taxa.csv", header=None)
# period_taxa = period_taxa.drop(index=0)

# # Merge entries in each row with a '.' separator
# # Drop the first row and merge entries in each row with a '.' separator
# merged_taxa = period_taxa.apply(lambda row: '.'.join(row.dropna().astype(str)), axis=1)
# # Create a DataFrame with merged_taxa as the index and period_taxa as the data
# period_taxa.index = merged_taxa


FileNotFoundError: [Errno 2] No such file or directory: '/home/kchen/microbiome/gut_microbiome_GPT/datasets/hmc_final/period_taxa.csv'

In [6]:

def find_inconsistent_taxonomy(taxon_df):
    """
    Find all rows where a lower-level taxon is identical across rows
    but its immediate parent is different.
    
    Returns a DataFrame with the offending child taxon and its distinct parents.
    """
    inconsistent_records = []

    ranks = taxon_df.columns.tolist()
    
    # Iterate over each lower-level rank (starting from second column)
    for i in range(1, len(ranks)):
        child = ranks[i]
        parent = ranks[i-1]
        
        # Group by child taxon
        grouped = taxon_df.groupby(child)[parent].nunique()
        # print(taxon_df.groupby(child)[parent].apply(list))
        
        # Child taxa with more than one unique parent
        problem_children = grouped[grouped > 1].index.tolist()
        
        for c in problem_children:
            parents = taxon_df.loc[taxon_df[child] == c, parent].unique().tolist()
            inconsistent_records.append({
                'Child_Rank': child,
                'Child_Taxon': c,
                'Parent_Rank': parent,
                'Distinct_Parents': parents
            })
    
    return pd.DataFrame(inconsistent_records)

# Apply to your DataFrame
taxon_df = data.varm['taxonomy']

inconsistent_df = find_inconsistent_taxonomy(taxon_df)
print(inconsistent_df)


   Child_Rank     Child_Taxon Parent_Rank  \
0      Phylum              NA      Domain   
1       Class  Incertae Sedis      Phylum   
2       Class              NA      Phylum   
3       Order  Incertae Sedis       Class   
4       Order              NA       Class   
5      Family  Incertae Sedis       Order   
6      Family              NA       Order   
7      Family  Unknown Family       Order   
8       Genus  Incertae Sedis      Family   
9       Genus              NA      Family   
10      Genus         UCG-001      Family   
11      Genus   endosymbionts      Family   

                                     Distinct_Parents  
0                  [Bacteria, NA, Eukaryota, Archaea]  
1   [Bacillota, Incertae Sedis, FCPU426, NKB15, SA...  
2   [Bacillota, NA, Pseudomonadota, Thermodesulfob...  
3   [Clostridia, Incertae Sedis, Gracilibacteria, ...  
4   [Clostridia, NA, Bacilli, Bacteroidia, Gammapr...  
5   [RF39, Clostridia UCG-014, Saccharimonadales, ...  
6   [NA, Enterobactera

In [ ]:
import sys, os

# Adjust this depending on where your notebook lives
sys.path.append(os.path.abspath(".."))  # two levels up
from data_utils.graph_helpers import *

graph_data = build_tg_data_from_taxon_df(taxon_df, taxon_df.index.tolist())

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Identified ['Archaea', 'Bacteria'] root nodes (no parents).
4369


In [ ]:
import numpy as np
import anndata as ad

# Convert tensor to numpy
vocabindex_to_nodeindex = graph_data.vocabindex_to_nodeindex.numpy()

# Find duplicates in node index
unique, counts = np.unique(vocabindex_to_nodeindex, return_counts=True)
duplicates = unique[counts > 1]

# Each duplicate value corresponds to multiple var_names (features)
duplicate_groups = [
    np.where(vocabindex_to_nodeindex == dup)[0]
    for dup in duplicates
]

print(f"Found {len(duplicate_groups)} duplicate feature groups.")
print(f"this is a total of {sum(len(g) for g in duplicate_groups)} duplicate features.")

# Get expression matrix
X = data.X

for group in duplicate_groups:
    if len(group) < 2:
        continue

    # These are the var indices corresponding to the duplicates
    dup_var_idxs = group
    dup_var_names = [data.var_names[i] for i in dup_var_idxs]

    # Extract expression submatrix for these vars
    submatrix = X[:, dup_var_idxs]

    # Count how many of these are nonzero per obs
    nonzero_counts = np.count_nonzero(submatrix, axis=1)
    conflict_obs = np.where(nonzero_counts > 1)[0]

    if len(conflict_obs) > 0:
        print(f"\nDuplicate features {dup_var_names}:")

    # ---- Merge duplicates (example: sum them) ----
    merged_col = np.sum(submatrix, axis=1)

    # Replace first duplicate col with merged values, zero others
    X[:, dup_var_idxs[0]] = merged_col
    for drop_idx in dup_var_idxs[1:]:
        X[:, drop_idx] = 0

# Optionally, drop redundant columns
keep_mask = np.ones(X.shape[1], dtype=bool)
for group in duplicate_groups:
    keep_mask[group[1:]] = False

adata_colmerged = ad.AnnData(
    X[:, keep_mask],
    obs=data.obs.copy(),
    var=data.var.iloc[keep_mask].copy()
)

print("✅ Duplicate columns merged and redundant ones removed.")


Found 369 duplicate feature groups.
this is a total of 753 duplicate features.
✅ Duplicate columns merged and redundant ones removed.


In [14]:
print(adata_colmerged)
print(data)

AnnData object with n_obs × n_vars = 105482 × 4296
    obs: 'drr', 'study_id', 'location'
    var: 'taxa'
AnnData object with n_obs × n_vars = 105482 × 4680
    obs: 'drr', 'study_id', 'location'
    var: 'taxa'
    obsm: 'raw_embedding'
    varm: 'taxonomy'
    layers: 'top_512'


In [10]:
hmc = np.load(DATA_DIR + "hmc_evo2_embeddings.npy")

In [12]:
hmc.shape

(4680, 4096)